# 🎮 Phase 3: NLP & Review Text Processing (Sentiment Analysis & Semantic Embeddings)

> **Mục tiêu của Notebook:**
> 1. **Sentiment Analysis:** Trích xuất cảm xúc từ text đánh giá (Review Title + Text) bằng VADER.
> 2. **Sentiment Validation & Correlation:** Đánh giá mối tương quan giữa Sentiment Compound Score với Rating thực tế (1 - 5 sao).
> 3. **Item Sentiment Aggregation:** Tổng hợp chỉ số cảm xúc cho từng tựa game (Tỷ lệ khen/chê, độ đồng thuận).
> 4. **Semantic Text Embeddings:** Biến đổi Metadata game (Title, Category, Store, Features, Description) thành vector ngữ nghĩa 384 chiều bằng `Sentence-Transformers (all-MiniLM-L6-v2)`.
> 5. **Content-Based Similarity Prototyping:** Thử nghiệm gợi ý Top-K game tương đồng theo khoảng cách Cosine trên không gian vector ngữ nghĩa.
> 6. **Visual Inspection:** Trực quan hóa không gian vector game bằng PCA/t-SNE.

In [ ]:
import os
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

# Thiết lập giao diện hiển thị biểu đồ chuẩn khoa học dữ liệu
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120

print("[*] Environment libraries loaded successfully!")

## 1. Load Clean Silver Datasets

Ta tải 2 tập dữ liệu sạch từ Silver Layer:
- `review_text.parquet`: Chứa 814,586 đánh giá kèm review text và rating.
- `item_features.parquet`: Chứa 25,612 game đầy đủ metadata.

In [ ]:
reviews_path = "../data/silver/review_text.parquet"
items_path = "../data/silver/item_features.parquet"

df_reviews = pl.read_parquet(reviews_path)
df_items = pl.read_parquet(items_path)

print(f"[+] Reviews count: {len(df_reviews):,}")
print(f"[+] Items count: {len(df_items):,}")
display(df_reviews.head(3).to_pandas())
display(df_items.head(3).to_pandas())

## 2. Sentiment Analysis with VADER

Thử nghiệm trích xuất Sentiment Score trên tập mẫu 50,000 reviews bằng `ReviewSentimentAnalyzer`.

In [ ]:
import sys
sys.path.append("..")
from src.nlp.sentiment import ReviewSentimentAnalyzer

analyzer = ReviewSentimentAnalyzer()

# Thử nghiệm trên một số câu review điển hình
sample_reviews = [
    "This game is an absolute masterpiece! The graphics and storyline are breathtaking.",
    "Terrible optimization, constant crashes and bugs. Total waste of money.",
    "It's an okay game. Nothing groundbreaking but good enough to kill some time.",
    "Not bad, but not worth the full price. Wait for a discount."
]

for r in sample_reviews:
    scores = analyzer.get_sentiment(r)
    label = analyzer.classify_compound_score(scores['compound'])
    print(f"Review: {r}")
    print(f"Scores: {scores} -> Label: {label}\n")

In [ ]:
# Chạy phân tích cảm xúc trên tập mẫu 50,000 reviews
df_scored = analyzer.process_reviews_dataframe(df_reviews, sample_size=50000)
df_scored_pd = df_scored.to_pandas()
display(df_scored_pd[['rating', 'title', 'sentiment_compound', 'sentiment_label']].head(5))

### 2.1. Phân tích tương quan giữa Sentiment Score và Star Rating

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Biểu đồ 1: Phân bố Compound Score theo từng mức Star Rating
sns.boxplot(x='rating', y='sentiment_compound', data=df_scored_pd, ax=axes[0], palette='Blues')
axes[0].set_title("Distribution of Sentiment Compound Score across Star Ratings", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Star Rating (1 - 5)")
axes[0].set_ylabel("Sentiment Compound Score (-1 to 1)")

# Biểu đồ 2: Tỷ lệ phân loại nhãn cảm xúc theo từng mức Star Rating
sentiment_dist = pd.crosstab(df_scored_pd['rating'], df_scored_pd['sentiment_label'], normalize='index') * 100
sentiment_dist[['positive', 'neutral', 'negative']].plot(kind='bar', stacked=True, ax=axes[1], color=['#2ecc71', '#95a5a6', '#e74c3c'])
axes[1].set_title("Sentiment Label Composition across Star Ratings (%)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Star Rating")
axes[1].set_ylabel("Percentage (%)")
axes[1].legend(title="Sentiment")

plt.tight_layout()
plt.show()

### 2.2. Tổng hợp chỉ số cảm xúc theo từng Game (Item Sentiment Profiles)

In [ ]:
df_item_sentiment = analyzer.aggregate_item_sentiments(df_scored)

# Merge thêm Game Title để quan sát
df_item_sentiment_with_title = df_item_sentiment.join(
    df_items.select(["parent_asin", "title", "main_category"]),
    on="parent_asin",
    how="left"
)

print("Top 5 game có điểm cảm xúc tích cực cao nhất (ít nhất 10 reviews trong tập mẫu):")
display(df_item_sentiment_with_title.filter(pl.col("review_count") >= 10).sort("avg_sentiment_compound", descending=True).head(5).to_pandas())

## 3. Semantic Text Embeddings with Sentence-Transformers

Mỗi tựa game được tổng hợp thành đoạn văn bản mô tả ngữ nghĩa (Title, Category, Store, Features, Description), sau đó mã hóa thành vector 384 chiều bằng mô hình `all-MiniLM-L6-v2`.

In [ ]:
from src.nlp.embeddings import ItemTextEmbedder

embedder = ItemTextEmbedder(model_name="all-MiniLM-L6-v2")

# Thử nghiệm trích xuất văn bản mô tả cho 5 game mẫu
sample_items = df_items.head(5)
sample_texts = embedder.generate_item_texts(sample_items)
for i, text in enumerate(sample_texts[:2]):
    print(f"--- Sample Game {i+1} Text Representation ---")
    print(text)
    print()

In [ ]:
# Trích xuất embeddings cho 500 game mẫu để thử nghiệm tính tương đồng
test_subset_items = df_items.head(500)
test_texts = embedder.generate_item_texts(test_subset_items)
test_embeddings = embedder.encode_texts(test_texts, batch_size=64, normalize_embeddings=True)

print(f"[+] Generated embeddings shape: {test_embeddings.shape}")

### 3.1. Thử nghiệm Content-Based Top-K Semantic Similarity Search

Kiểm tra khả năng tìm kiếm game tương đồng dựa trên Cosine Similarity giữa các vector embeddings.

In [ ]:
# Tính ma trận tương đồng Cosine Similarity
sim_matrix = cosine_similarity(test_embeddings)

def get_top_k_similar_games(item_idx: int, top_k: int = 5):
    query_title = test_subset_items['title'][item_idx]
    sim_scores = list(enumerate(sim_matrix[item_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    print(f"🎮 Gốc: [{test_subset_items['parent_asin'][item_idx]}] {query_title}")
    print("=" * 75)
    print(f"{'Rank':<5} | {'Cosine Sim':<12} | {'Parent ASIN':<15} | {'Title'}")
    print("-" * 75)
    
    for rank, (idx, score) in enumerate(sim_scores[1:top_k+1], 1):
        title = test_subset_items['title'][idx]
        asin = test_subset_items['parent_asin'][idx]
        print(f"{rank:<5} | {score:.4f}{'':<6} | {asin:<15} | {title}")

# Kiểm tra thử với game ở index 0 và index 10
get_top_k_similar_games(0, top_k=5)
print()
get_top_k_similar_games(10, top_k=5)

### 3.2. Trực quan hóa Không gian Vector Embeddings bằng PCA 2D

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(test_embeddings)

plt.figure(figsize=(12, 7))
scatter = plt.scatter(
    coords_2d[:, 0],
    coords_2d[:, 1],
    c=test_subset_items['average_rating'].to_numpy(),
    cmap='viridis',
    alpha=0.8,
    s=40,
    edgecolor='none'
)
plt.colorbar(scatter, label='Average Rating')
plt.title("2D PCA Projection of Video Game Semantic Embeddings (dim: 384 -> 2)", fontsize=14, fontweight='bold')
plt.xlabel(f"Principal Component 1 (Var: {pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"Principal Component 2 (Var: {pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.tight_layout()
plt.show()

## 4. Kết luận & Kế hoạch đóng gói Gold Layer

- **Sentiment Signals:** Điểm Sentiment Compound từ VADER tương quan thuận rất tốt với điểm Star Rating (1 sao chủ yếu negative, 5 sao >80% positive). Điểm này sẽ là thành phần quan trọng trong việc chấm điểm xếp hạng (Ranking) và cung cấp lý do giải thích gợi ý (Explanation Signals).
- **Semantic Embeddings:** Mô hình `all-MiniLM-L6-v2` cho ra các vector đại diện ngữ nghĩa chính xác, hỗ trợ tìm kiếm game tương đồng và Content-Based Filtering hiệu quả.
- **Module hoàn thiện:**
  - `src/nlp/sentiment.py`: Pipeline phân tích và tổng hợp cảm xúc game.
  - `src/nlp/embeddings.py`: Pipeline trích xuất vector embeddings 384-D và xuất Gold Layer.